# ACTO SDK: Fleet Management

This notebook demonstrates how to use the Fleet Management API to monitor and organize your robot fleet.


In [ ]:
import httpx
import json

# Configuration
BASE_URL = "http://127.0.0.1:8080"  # Local server
JWT_TOKEN = "your-jwt-token-here"  # Get this from wallet authentication

headers = {
    "Authorization": f"Bearer {JWT_TOKEN}",
    "Content-Type": "application/json",
}


## Fleet Overview

Get a summary of all devices in your fleet.


In [ ]:
response = httpx.get(f"{BASE_URL}/v1/fleet", headers=headers, timeout=30)
fleet = response.json()

print("Fleet Summary:")
print(f"  Total devices: {fleet['summary']['total_devices']}")
print(f"  Active devices: {fleet['summary']['active_devices']}")
print(f"  Total groups: {fleet['summary']['total_groups']}")

print("\nDevices:")
for device in fleet['devices']:
    status_emoji = "🟢" if device['status'] == 'active' else "🔴" if device['status'] == 'offline' else "🟡"
    print(f"  {status_emoji} {device['name']} ({device['id']})")


## Device Details

Get detailed information about a specific device including health metrics and activity logs.


In [ ]:
# Replace with an actual device ID from your fleet
device_id = "robot-001"

response = httpx.get(
    f"{BASE_URL}/v1/fleet/devices/{device_id}",
    headers=headers,
    timeout=30
)
details = response.json()

print(f"Device: {details['name']}")
print(f"Status: {details['status']}")
print(f"Proof count: {details['proof_count']}")
print(f"Task count: {details['task_count']}")

if details.get('health'):
    health = details['health']
    print("\nHealth Metrics:")
    if health.get('cpu_percent'):
        print(f"  CPU: {health['cpu_percent']}%")
    if health.get('memory_percent'):
        print(f"  Memory: {health['memory_percent']}%")
    if health.get('battery_percent'):
        print(f"  Battery: {health['battery_percent']}%")


## Rename Device

Give your robots custom, human-readable names.


In [ ]:
device_id = "robot-001"
new_name = "Warehouse Robot Alpha"

response = httpx.patch(
    f"{BASE_URL}/v1/fleet/devices/{device_id}/name",
    headers=headers,
    json={"name": new_name},
    timeout=30
)
result = response.json()

print(f"Device renamed to: {result['name']}")


## Report Device Health

Report health metrics from your robot. All fields are optional - only send what your device supports.


In [ ]:
device_id = "robot-001"

# All fields are optional - include only what your device provides
health_payload = {
    "cpu_percent": 45.2,
    "memory_percent": 68.0,
    "battery_percent": 85.0,
    "battery_charging": True,
    "disk_percent": 23.5,
    "temperature": 42.5,
    "uptime_seconds": 86400,
    "network_connected": True,
    # Custom metrics for device-specific data
    "custom_metrics": {
        "lidar_status": "ok",
        "motor_temp_left": 38.2,
        "motor_temp_right": 39.1,
    }
}

response = httpx.post(
    f"{BASE_URL}/v1/fleet/devices/{device_id}/health",
    headers=headers,
    json=health_payload,
    timeout=30
)
result = response.json()

print(f"Health reported: {result['success']}")
print(f"Record ID: {result['record_id']}")


## Device Groups

Organize your robots into logical groups (e.g., "Warehouse A", "Production Line").


In [ ]:
# Create a new group
response = httpx.post(
    f"{BASE_URL}/v1/fleet/groups",
    headers=headers,
    json={
        "name": "Warehouse A",
        "description": "Main warehouse robots",
        "color": "#22c55e"  # Green
    },
    timeout=30
)
result = response.json()
group_id = result['group']['id']

print(f"Created group: {result['group']['name']}")
print(f"Group ID: {group_id}")


In [ ]:
# List all groups
response = httpx.get(f"{BASE_URL}/v1/fleet/groups", headers=headers, timeout=30)
groups = response.json()

print("Device Groups:")
for group in groups['groups']:
    print(f"  📁 {group['name']} - {group.get('description', 'No description')}")


In [ ]:
# Assign devices to a group
response = httpx.post(
    f"{BASE_URL}/v1/fleet/groups/{group_id}/assign",
    headers=headers,
    json={"device_ids": ["robot-001", "robot-002"]},
    timeout=30
)
result = response.json()

print(f"Assigned {len(result['assigned'])} devices to group")


## WebSocket for Real-time Updates

Connect to the WebSocket endpoint for live device status updates.


In [ ]:
import asyncio

# Note: This requires the websockets package: pip install websockets
# Uncomment the code below to run

"""
import websockets

async def listen_fleet_updates():
    uri = f"ws://127.0.0.1:8080/ws/fleet?token={JWT_TOKEN}"
    
    async with websockets.connect(uri) as ws:
        print("Connected to fleet WebSocket")
        
        try:
            while True:
                message = await asyncio.wait_for(ws.recv(), timeout=30)
                data = json.loads(message)
                
                event_type = data.get('type')
                if event_type == 'device_status':
                    print(f"Device {data['device_id']} is now {data['status']}")
                elif event_type == 'health_update':
                    print(f"Health update for {data['device_id']}")
                elif event_type == 'proof_submitted':
                    print(f"New proof from {data['device_id']}")
                else:
                    print(f"Event: {data}")
                    
        except asyncio.TimeoutError:
            print("No updates for 30 seconds, disconnecting")

await listen_fleet_updates()
"""

print("WebSocket example (see code comments to enable)")


## Cleanup

Delete the test group.


In [ ]:
response = httpx.delete(
    f"{BASE_URL}/v1/fleet/groups/{group_id}",
    headers=headers,
    timeout=30
)
result = response.json()

print(f"Group deleted: {result['success']}")
